# 04 · Liquidation math: 5x / 10x / 20x longs through the 10/10 crash

Module **M6: liquidations and cascades** (Phase 0).

**Task.** Implement an approximate liquidation-price function for linear isolated positions,
unit-test it against hand-computed cases, and simulate how many hypothetical 5x/10x/20x longs
opened on Oct 9, 2025 would have been liquidated using real 1-minute candles.

**Market.** Binance USDⓈ-M perpetual **BTCUSDT** (ccxt: exchange `binanceusdm`, symbol
`BTC/USDT:USDT`). For comparison only: Binance **spot** BTC/USDT (exchange `binance`).

**Hypothesis (lab rule 1).** *I believe most 10x and 20x longs opened on Oct 9 were liquidated on
Oct 10, because with leverage L a drop of roughly 1/L (10% and 5%) wipes out the margin, and the
crash of Oct 10 was larger than that. I believe 5x longs mostly survived, because they need a drop
of about 20%. I believe liquidations measured with the mark price are fewer than with the last
price, because the mark price is built to ignore short wicks on one exchange.*
Honesty note: this hypothesis was written **after** Step 0 had shown the minimum price of the
window, so it is not blind.

**Glossary (new terms).**

- **Perpetual (perp)**: a futures contract with no expiry date. Its price is kept close to spot
  by **funding** payments between longs and shorts (module M7).
- **Linear contract**: quoted and settled in USDT. PnL = `qty × (price − entry)` USDT for a long.
- **Leverage (L)**: notional ÷ margin. 1,000 USDT of margin at 10x controls 10,000 USDT of BTC.
- **Isolated margin**: only the margin posted for this position can be lost.
- **Notional**: `qty × price`, the USDT value of the position.
- **Maintenance margin rate (MMR)**: the fraction of the notional the position must keep as
  equity. Below it, the exchange closes the position: **liquidation**.
- **Liquidation price (LP)**: the price where equity = maintenance margin.
- **Last price**: the price of the latest trade on this exchange (normal candles).
- **Mark price**: a price the exchange **computes** (from an index of spot prices on several
  exchanges, plus a funding-based term) and uses to decide liquidations, so that one wild trade
  on its own book does not liquidate everyone.
- **Wick**: the part of a candle between its low (or high) and its body; a short spike.
- **Cascade**: liquidations sell into the market, pushing the price down, which triggers more
  liquidations.

## 1. Setup and parameters

**Parameters fixed before looking at the data** (lab rule 4). Changes after inspection are
labeled as such.

- Window (UTC, start included, end excluded): **2025-10-09 00:00 → 2025-10-11 00:00**, 2 days =
  2,880 one-minute candles per series.
- Three series, all 1-minute: **perp last** (`fetch_ohlcv`, `GET /fapi/v1/klines`), **perp mark**
  (`fetch_ohlcv` with `params={"price": "mark"}`, `GET /fapi/v1/markPriceKlines`) and **spot last**
  (Binance spot, `GET /api/v3/klines`).
- Simulation: one isolated long per minute of 2025-10-09 (00:00–23:59, 1,440 entries) for each
  leverage **5x, 10x, 20x**; margin **1,000 USDT**; entry = **close** of the perp last candle of
  that minute; `qty = margin × L / entry`.
- **MMR = 0.40%, cum = 0** (tier 1 of BTCUSDT, notional 0–300,000 USDT). See the assumption in
  Step 1.

In [1]:
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.bitstamp import load_or_fetch_json
from src.data.candles import to_utc, validate_candles
from src.data.ohlcv import (cache_path, fetch_ohlcv_window, fetch_page_with_retry,
                            load_or_fetch_window, make_exchange)
from src.liquidation import (approx_liquidation_price, equity_and_maintenance,
                             liquidation_price_isolated, simulate_isolated_longs)

START, END = to_utc("2025-10-09 00:00"), to_utc("2025-10-11 00:00")
ENTRY_DAY = ("2025-10-09 00:00", "2025-10-09 23:59")
SERIES = {  # name: (ccxt exchange id, symbol, price type)
    "perp last": ("binanceusdm", "BTC/USDT:USDT", None),
    "perp mark": ("binanceusdm", "BTC/USDT:USDT", "mark"),
    "spot last": ("binance", "BTC/USDT", None),
}
LEVERAGES = [5, 10, 20]
MARGIN = 1_000.0       # USDT per position
MMR, CUM = 0.004, 0.0  # tier 1 of BTCUSDT (assumption, see Step 1)
TIER1_MAX_NOTIONAL = 300_000.0
CACHE_DIR = REPO_ROOT / "data" / "raw" / "04_liquidation_math"

pd.set_option("display.width", 120)

## 2. Step 0 · Check the sources before using them

The documentation (read on 2026-11-16) says, for both `GET /fapi/v1/klines` and
`GET /fapi/v1/markPriceKlines`: max `limit` 1500, default 500, `startTime`/`endTime` in ms, oldest
first. Docs:
[Kline](https://developers.binance.com/docs/derivatives/usds-margined-futures/market-data/rest-api/Kline-Candlestick-Data),
[Mark Price Kline](https://developers.binance.com/docs/derivatives/usds-margined-futures/market-data/rest-api/Mark-Price-Kline-Candlestick-Data).

We **measure** instead of trusting them, both on the raw API (ccxt's `fapiPublicGet...` methods,
which send our parameters unchanged) and through ccxt's `fetch_ohlcv`. Results are cached in
`data/`; delete the cache file to measure again.

- `raw API max (asked → got)`: we ask for 1500 and 2000 candles. An error means the API rejects the
  value instead of capping it.
- `ccxt max (asked → got)`: the same through `fetch_ohlcv`, which is what our download uses.
- `field 5 (volume)` and `field 8`: the distinct values in one page of 1,500 raw rows.

In [2]:
RAW_METHOD = {"perp last": "fapiPublicGetKlines", "perp mark": "fapiPublicGetMarkPriceKlines"}


def check_perp_source(name):
    try:
        ex = make_exchange("binanceusdm")
        _, symbol, price = SERIES[name]
        since = int(START.timestamp() * 1000)
        raw = getattr(ex, RAW_METHOD[name])
        raw_caps = {}
        for limit in (None, 1500, 2000):
            params = {"symbol": "BTCUSDT", "interval": "1m", "startTime": since}
            if limit:
                params["limit"] = limit
            try:
                raw_caps[limit] = len(raw(params))
            except Exception as error:
                raw_caps[limit] = f"error: {str(error)[-60:]}"
        page = raw({"symbol": "BTCUSDT", "interval": "1m", "startTime": since, "limit": 1500})
        ccxt_caps = {limit: len(fetch_page_with_retry(ex, symbol, "1m", since, limit, price=price))
                     for limit in (None, 1000, 1500, 2000)}
        stamps = [row[0] for row in page]
        first = fetch_page_with_retry(ex, symbol, "1m", int(to_utc("2017-01-01").timestamp() * 1000),
                                      1, price=price)[0][0]
        now_ms = ex.milliseconds()
        newest = fetch_page_with_retry(ex, symbol, "1m", None, 3, price=price)[-1][0]
        return {
            "default page (raw)": raw_caps[None],
            "raw API max (asked → got)": f"1500→{raw_caps[1500]}, 2000→{raw_caps[2000]}",
            "ccxt max (asked → got)": ", ".join(f"{k}→{v}" for k, v in ccxt_caps.items() if k),
            "raw first timestamp": stamps[0],
            "parsed (ms, UTC)": str(pd.to_datetime(stamps[0], unit="ms", utc=True)),
            "oldest first": stamps == sorted(stamps),
            "field 5 (volume) values": sorted({row[5] for row in page})[:3] if name == "perp mark"
                                       else f"{len({row[5] for row in page})} distinct, min {min(float(r[5]) for r in page)}",
            "field 8 values (count)": ", ".join(
                f"{k}: {n} rows" for k, n in pd.Series([row[8] for row in page]).value_counts().head(3).items())
                if name == "perp mark" else "number of trades",
            "first 1m candle (UTC)": str(pd.to_datetime(first, unit="ms", utc=True)),
            "last candle still forming": newest + 60_000 > now_ms,
            "checked at (UTC)": str(pd.to_datetime(now_ms, unit="ms", utc=True).floor("s")),
        }
    except Exception as error:  # report, do not hide
        return {"access": f"{type(error).__name__}: {str(error)[:120]}"}


source_checks = load_or_fetch_json(CACHE_DIR / "source_checks.json",
                                   lambda: {n: check_perp_source(n) for n in RAW_METHOD})
pd.DataFrame(source_checks)

,perp last,perp mark
default page (raw),500,500
raw API max (asked → got),"1500→1500, 2000→error: -1130,""msg"":""Data sent ...","1500→1500, 2000→error: -1130,""msg"":""Data sent ..."
ccxt max (asked → got),"1000→1000, 1500→1000, 2000→1000","1000→1000, 1500→1000, 2000→1000"
raw first timestamp,1759968000000,1759968000000
"parsed (ms, UTC)",2025-10-09 00:00:00+00:00,2025-10-09 00:00:00+00:00
oldest first,True,True
field 5 (volume) values,"1488 distinct, min 3.143",[0]
field 8 values (count),number of trades,"60: 1498 rows, 59: 2 rows"
first 1m candle (UTC),2019-09-08 17:57:00+00:00,2019-12-23 11:58:00+00:00
last candle still forming,True,True


**Technical sheet (same format as notebook 03).**

| | Binance spot (notebook 03) | Binance USDⓈ-M perp, last | Binance USDⓈ-M perp, mark |
|---|---|---|---|
| Max per request | 1000 | **API accepts 1500; ccxt caps at 1000** (2000 → error −1130) | **API accepts 1500; ccxt caps at 1000** |
| Default | 500 | 500 | 500 |
| Timestamp | ms, open time, UTC | ms, open time, UTC | ms, open time, UTC |
| Order | oldest first | oldest first | oldest first |
| First 1m candle | 2017-08-17 04:00 UTC | 2019-09-08 17:57 UTC | 2019-12-23 11:58 UTC |
| Last candle | still forming → dropped | still forming → dropped | still forming → dropped |
| Volume | traded BTC | traded BTC | **always 0**: the mark is a computed price, nothing trades at it |
| Field 8 | number of trades | number of trades | **Ignore according to the docs** (it is 60 almost always); not used |

We download with ccxt, 1000 candles per request (`EXCHANGE_MAX_LIMIT["binanceusdm"]` in
`src/data/ohlcv.py`). Last and mark are cached in **separate files** whose name ends in the price
type (`..._last.parquet`, `..._mark.parquet`), so one series can never be read as the other.

### 2.1 Download, coverage and quality

`fetch_ohlcv_window` downloads candles with open time in `[start, end)`, removes duplicates and the
candle still forming, and never fills a gap. The checks below stop the notebook if anything is off.

In [3]:
frames, infos = {}, {}
for name, (ex_id, symbol, price) in SERIES.items():
    path = cache_path(CACHE_DIR, ex_id, symbol, "1m", "oct9-10", price=price)
    frames[name], infos[name] = load_or_fetch_window(
        path, lambda: fetch_ohlcv_window(ex_id, symbol, START, END, price=price))

for name, df in frames.items():
    assert str(df.index.tz) == "UTC", name
    assert df.index.is_unique and df.index.is_monotonic_increasing, name
    assert df.index.min() >= START and df.index.max() < END, name
    assert (df[["open", "high", "low", "close"]] > 0).all().all(), name
    assert (df["high"] >= df[["open", "close", "low"]].max(axis=1)).all(), name
    assert (df["low"] <= df[["open", "close", "high"]].min(axis=1)).all(), name

last, mark, spot = frames["perp last"], frames["perp mark"], frames["spot last"]


def coverage_row(name, df):
    report = validate_candles(df, START, END - pd.Timedelta(minutes=1))
    i_low = df["low"].idxmin()
    return {"candles": report["candles"], "expected": report["expected"],
            "missing": report["missing"], "duplicates dropped": infos[name]["duplicates_dropped"],
            "volume == 0": int((df["volume"] == 0).sum()),
            "O=H=L=C": int(((df["open"] == df["high"]) & (df["high"] == df["low"])
                            & (df["low"] == df["close"])).sum()),
            "min low": round(df["low"].min(), 2), "min low at (UTC)": f"{i_low:%m-%d %H:%M}"}


coverage = pd.DataFrame({name: coverage_row(name, df) for name, df in frames.items()}).T
coverage

,candles,expected,missing,duplicates dropped,volume == 0,O=H=L=C,min low,min low at (UTC)
perp last,2880,2880,0,0,0,0,101516.5,10-10 21:20
perp mark,2880,2880,0,0,2880,0,101877.69,10-10 21:20
spot last,2880,2880,0,0,0,0,102000.0,10-10 21:20


How to read it: one row per series. `volume == 0` is a filled candle for a traded series; for the
**mark** it is every candle, because the mark has no volume (not a quality problem). `O=H=L=C`
counts candles that are flat (a typical filled candle). `min low` is the lowest price of the
2 days.

### 2.2 Quick comparison: perp vs spot, last vs mark, calm day vs crash day

Differences in bps, `(a / b − 1) × 10,000`, minute by minute, using the `close` or the `low` of the
same minute. Oct 9 is the calm day; Oct 10 contains the crash.

In [4]:
COMPARISONS = {  # label: (series a, series b, column)
    "close perp − spot": ("perp last", "spot last", "close"),
    "low perp − spot": ("perp last", "spot last", "low"),
    "close last − mark": ("perp last", "perp mark", "close"),
    "low last − mark": ("perp last", "perp mark", "low"),
}
# Added after inspecting this table (see the note below it).
NOTES = {("2025-10-10", "low perp − spot"):
         "max: one single minute, caused by the spot wick; it does not mean the perp was ~5% more expensive"}

rows = {}
for day in ["2025-10-09", "2025-10-10"]:
    for label, (a, b, col) in COMPARISONS.items():
        d = (10_000 * (frames[a][col] / frames[b][col] - 1)).loc[day]
        rows[day, label] = {"minutes": len(d), "median (bps)": round(d.median(), 1),
                            "p95 |diff| (bps)": round(d.abs().quantile(0.95), 1),
                            "min (bps)": round(d.min(), 1), "min at": f"{d.idxmin():%H:%M}",
                            "max (bps)": round(d.max(), 1), "max at": f"{d.idxmax():%H:%M}",
                            "note": NOTES.get((day, label), "")}
comparison = pd.DataFrame(rows).T
comparison.index.names = ["day (UTC)", "difference"]
comparison

minutes median (bps) p95 |diff| (bps) min (bps) min at max (bps) max at  \
day (UTC)  difference                                                                                  
2025-10-09 close perp − spot    1440         -4.6              5.7      -6.8  23:59      -2.6  13:22   
           low perp − spot      1440         -4.8              6.1     -10.7  14:46      -1.0  13:30   
           close last − mark    1440         -0.0              1.9      -7.5  13:51       4.5  14:46   
           low last − mark      1440         -0.2              1.9      -6.6  14:46       2.4  03:28   
2025-10-10 close perp − spot    1440         -4.6              6.4     -51.4  21:19      77.1  21:27   
           low perp − spot      1440         -4.9              7.5    -154.8  21:19     527.1  21:24   
           close last − mark    1440         -0.0              4.1     -91.2  21:19      50.2  21:27   
           low last − mark      1440         -0.3              5.3    -132.8  21:19      27.4  21:32   

                                                                           note  
day (UTC)  difference                                                            
2025-10-09 close perp − spot                                                     
           low perp − spot                                                       
           close last − mark                                                     
           low last − mark                                                       
2025-10-10 close perp − spot                                                     
           low perp − spot    max: one single minute, caused by the spot wic...  
           close last − mark                                                     
           low last − mark

**A real wick on a thin book (spot, 2025-10-10 21:24 UTC).** The spot candle of 21:24 opened at
107,571 and closed at 108,219, but its low was **102,000.0**, exactly the same round number as the
spot minimum of 21:20. Neither the perp last nor the mark went near it. The same candle is in
notebook 03's cache, so it is Binance's data, not a download error. A likely reading: one sell
order swept a thin book down to a large buy order resting at 102,000. We keep it as it is; it does
not enter the simulation, which uses only the perp.

**Observation, not explained here.** The perp traded about **4.6 bps below spot** (median) on both
days, calm or not. This is revisited in **M7 (funding)**.

In [5]:
wick = pd.concat({"spot": spot[["open", "low", "close"]], "perp last": last[["low", "close"]],
                  "perp mark": mark[["low", "close"]]}, axis=1)
wick.loc["2025-10-10 21:19":"2025-10-10 21:25"].round(1)

spot                     perp last           perp mark          
                               open       low     close       low     close       low     close
open_time_utc                                                                                  
2025-10-10 21:19:00+00:00  108822.0  103800.0  103975.3  102193.6  103441.3  103568.5  104393.2
2025-10-10 21:20:00+00:00  104113.8  102000.0  104384.6  101516.5  104311.2  101877.7  104386.2
2025-10-10 21:21:00+00:00  104411.8  103255.1  106766.1  104212.8  106416.5  104267.8  107006.9
2025-10-10 21:22:00+00:00  106767.7  106252.9  107196.0  106375.4  107274.1  106750.2  107213.2
2025-10-10 21:23:00+00:00  107193.0  106461.5  107478.8  107202.5  107747.3  107292.5  107676.4
2025-10-10 21:24:00+00:00  107571.2  102000.0  108219.1  107376.9  108473.0  107266.9  108677.9
2025-10-10 21:25:00+00:00  108219.1  106388.6  106891.5  106982.9  107111.6  107131.8  107131.8

## 3. Step 1 · The liquidation price

Code in `src/liquidation.py`, tests in `tests/test_liquidation.py`.

The exchange liquidates an isolated position when its **equity** (margin + unrealized PnL) falls
to its **maintenance margin** (`notional × MMR − cum`, with notional at the current price).
Solving that equality for the price gives Binance's formula for isolated, one-way mode (wallet
balance = isolated margin, other positions = 0):

$$\text{long: } LP = \frac{\text{margin} + \text{cum} - \text{qty}\cdot\text{entry}}{\text{qty}\cdot\text{MMR} - \text{qty}}
\qquad
\text{short: } LP = \frac{\text{margin} + \text{cum} + \text{qty}\cdot\text{entry}}{\text{qty}\cdot\text{MMR} + \text{qty}}$$

Sources: [liquidation price](https://www.binance.com/en/support/faq/b3c689c1f50a44cabb3a84e663b81d93),
[maintenance margin = notional × MMR − cum](https://www.binance.com/en/support/faq/360033162192).

**The shortcut.** With leverage L, the margin is 1/L of the notional, so a drop of about 1/L wipes
it out; the exchange steps in a little earlier, when only MMR is left:
`LP ≈ entry × (1 − 1/L + MMR)` for a long, `entry × (1 + 1/L − MMR)` for a short.

**ASSUMPTION — MMR = 0.40%, cum = 0.** Tier 1 of BTCUSDT (notional 0–300,000 USDT) after Binance's
update of 2025-08-17/18
([announcement](https://www.binance.com/en/support/announcement/detail/8e428625ebeb4cc7ae8678026846095c)).
No change for BTCUSDT between that date and 2025-10-09 was found, but this is **not confirmed**.
All simulated positions must fall in tier 1 (checked below).

The table recomputes the unit-test cases (hand values computed in the chat before coding).
Case 6 checks the identity "equity = maintenance margin at LP", where the PnL is
`qty × (LP − entry)` for a long and `qty × (entry − LP)` for a short.

In [6]:
E, Q = 100_000.0, 0.1
cases = [
    ("1 long, margin 1,000, MMR 0.50%", 90_452.26, liquidation_price_isolated("long", E, Q, 1_000, 0.005)),
    ("2 long, margin 1,000, MMR 0.40%", 90_361.45, liquidation_price_isolated("long", E, Q, 1_000, 0.004)),
    ("3 short, margin 1,000, MMR 0.40%", 109_561.75, liquidation_price_isolated("short", E, Q, 1_000, 0.004)),
    ("4 long 100x, margin 100, MMR 0.40%", 99_397.59, liquidation_price_isolated("long", E, Q, 100, 0.004)),
    ("5 shortcut of case 1 (10x)", 90_500.00, approx_liquidation_price("long", E, 10, 0.005)),
]
tests = pd.DataFrame([{"case": c, "by hand": h, "function": round(f, 4), "diff": round(f - h, 4),
                       "pass (±0.01)": abs(f - h) <= 0.01} for c, h, f in cases]).set_index("case")
gap_pct = 100 * (cases[4][2] - cases[0][2]) / cases[0][2]
for side in ["long", "short"]:
    lp = liquidation_price_isolated(side, E, Q, 1_000, MMR)
    equity, maint = equity_and_maintenance(side, lp, E, Q, 1_000, MMR)
    tests.loc[f"6 identity at LP, {side} (equity vs maintenance)"] = [round(maint, 4), round(equity, 4),
                                                                      round(equity - maint, 6),
                                                                      abs(equity - maint) <= 0.01]
print(f"shortcut vs exact (case 5 vs case 1): {gap_pct:.3f}% above the exact LP")
assert tests["pass (±0.01)"].all() and gap_pct < 0.1
tests

shortcut vs exact (case 5 vs case 1): 0.053% above the exact LP


,by hand,function,diff,pass (±0.01)
case,,,,
"1 long, margin 1,000, MMR 0.50%",90452.2600,90452.2613,0.0013,True
"2 long, margin 1,000, MMR 0.40%",90361.4500,90361.4458,-0.0042,True
"3 short, margin 1,000, MMR 0.40%",109561.7500,109561.7530,0.0030,True
"4 long 100x, margin 100, MMR 0.40%",99397.5900,99397.5904,0.0004,True
5 shortcut of case 1 (10x),90500.0000,90500.0000,0.0000,True
"6 identity at LP, long (equity vs maintenance)",36.1446,36.1446,-0.0000,True
"6 identity at LP, short (equity vs maintenance)",43.8247,43.8247,-0.0000,True


## 4. Step 2 · Simulation: longs opened on Oct 9

For every minute `t` of 2025-10-09 and every leverage L:

1. **Entry** = close of the perp last candle of minute `t`; `qty = 1,000 × L / entry`.
2. **LP** = exact isolated formula with MMR 0.40%, cum 0.
3. **No look-ahead:** liquidation is checked from the minute **after** `t` up to the last minute
   before 2025-10-11 00:00. The low of minute `t` itself happened before the close we enter at.
4. **Liquidated (main)** = first minute whose **mark** low is ≤ LP. **Comparison** = the same with
   the **last** low.

Approximations, stated up front:

- The **low of a 1-minute mark candle** stands in for the real mark price, which Binance updates
  more often than once a minute. A touch inside the minute is caught; the exact second is not.
- **Fees and funding are ignored.** The opening fee would reduce the margin and move LP **closer**
  to the entry, so the real liquidation counts can only be equal or higher. Funding over the
  ~1–2 days held could go either way.
- No liquidation fee, no partial liquidation, no slippage when the position is closed: we only
  ask **whether** the LP was reached, not what the trader got back (with isolated margin it is at
  most the 1,000 USDT lost).

In [7]:
entry_close = last.loc[ENTRY_DAY[0]:ENTRY_DAY[1], "close"]
assert len(entry_close) == 1_440, len(entry_close)
for name in ["perp last", "perp mark"]:  # the checks need every minute after each entry
    assert validate_candles(frames[name], START, END - pd.Timedelta(minutes=1))["missing"] == 0, name

sim = simulate_isolated_longs(entry_close, {"mark": mark["low"], "last": last["low"]},
                              LEVERAGES, MARGIN, MMR, CUM)
assert (sim["notional"] <= TIER1_MAX_NOTIONAL).all()   # every position is in tier 1
entry_times = sim.index.get_level_values("entry_time")
for price in ["mark", "last"]:  # a liquidation can never be at or before its own entry minute
    assert (entry_times.to_numpy() < sim[f"liquidated_{price}"].fillna(END).to_numpy()).all()
print(f"{len(sim):,} positions; notional per leverage (USDT):",
      sim.groupby(level="leverage")["notional"].max().round(2).to_dict())
sim.head()

4,320 positions; notional per leverage (USDT): {5: 5000.0, 10: 10000.0, 20: 20000.0}


entry       qty  notional            lp  drop_to_lp_pct liquidated_mark  \
leverage entry_time                                                                                              
5        2025-10-09 00:00:00+00:00  123272.9  0.040560    5000.0  99014.377510       19.678715             NaT   
         2025-10-09 00:01:00+00:00  123242.8  0.040570    5000.0  98990.200803       19.678715             NaT   
         2025-10-09 00:02:00+00:00  123214.0  0.040580    5000.0  98967.068273       19.678715             NaT   
         2025-10-09 00:03:00+00:00  123208.0  0.040582    5000.0  98962.248996       19.678715             NaT   
         2025-10-09 00:04:00+00:00  123156.6  0.040599    5000.0  98920.963855       19.678715             NaT   

                                   liquidated_last  
leverage entry_time                                 
5        2025-10-09 00:00:00+00:00             NaT  
         2025-10-09 00:01:00+00:00             NaT  
         2025-10-09 00:02:00+00:00             NaT  
         2025-10-09 00:03:00+00:00             NaT  
         2025-10-09 00:04:00+00:00             NaT

### T1 · How many were liquidated?

Per leverage: share of the 1,440 positions liquidated by the mark (main) and by the last
(comparison), the median drop from entry to LP, and the median UTC time of liquidation among the
liquidated ones.

In [8]:
def fmt(ts):
    return f"{ts:%m-%d %H:%M}"


def t1_row(group):
    row = {"positions": len(group),
           "LP drop from entry, median (%)": round(group["drop_to_lp_pct"].median(), 2),
           "shortcut drop 1/L − MMR (%)": round(100 * (1 / group.name - MMR), 2)}
    for price in ["mark", "last"]:
        hit = group[f"liquidated_{price}"].dropna()
        row[f"liquidated, {price} (%)"] = round(100 * len(hit) / len(group), 1)
        row[f"median liq. time, {price} (UTC)"] = fmt(hit.median()) if len(hit) else "—"
    # Added after inspecting the data: the median alone hid how spread out the times were.
    # p10/p90 use interpolation="nearest", so they are real minutes of the sample.
    hit = group["liquidated_mark"].dropna()
    for label, value in [("first", hit.min() if len(hit) else None),
                         ("p10", hit.quantile(0.10, interpolation="nearest") if len(hit) else None),
                         ("p90", hit.quantile(0.90, interpolation="nearest") if len(hit) else None),
                         ("last", hit.max() if len(hit) else None)]:
        row[f"[post-inspection] {label} liq. time, mark (UTC)"] = fmt(value) if value is not None else "—"
    return pd.Series(row)


t1 = sim.groupby(level="leverage").apply(t1_row)
t1.T

leverage,5,10,20
positions,1440,1440,1440
"LP drop from entry, median (%)",19.68,9.64,4.62
shortcut drop 1/L − MMR (%),19.6,9.6,4.6
"liquidated, mark (%)",0.0,100.0,100.0
"median liq. time, mark (UTC)",—,10-10 21:13,10-10 19:21
"liquidated, last (%)",0.0,100.0,100.0
"median liq. time, last (UTC)",—,10-10 21:13,10-10 19:21
"[post-inspection] first liq. time, mark (UTC)",—,10-10 21:13,10-10 17:05
"[post-inspection] p10 liq. time, mark (UTC)",—,10-10 21:13,10-10 18:52
"[post-inspection] p90 liq. time, mark (UTC)",—,10-10 21:17,10-10 20:54


**Added after inspecting the data.** Which exact minutes hold the 10x liquidations, and are they
inside 20:50–21:30 UTC of Oct 10 (the window shaded in G1)? Count of positions per minute, by mark
(main) and by last (comparison).

In [9]:
CRASH_WINDOW = (to_utc("2025-10-10 20:50"), to_utc("2025-10-10 21:30"))  # end included
ten = sim.xs(10, level="leverage")
minutes_10x = pd.DataFrame({price: ten[f"liquidated_{price}"].value_counts()
                            for price in ["mark", "last"]}).fillna(0).astype(int).sort_index()
minutes_10x.index = minutes_10x.index.strftime("%m-%d %H:%M")
minutes_10x.loc["total"] = minutes_10x.sum()
for price in ["mark", "last"]:
    t = ten[f"liquidated_{price}"]
    inside = t.between(*CRASH_WINDOW).sum()
    print(f"10x by {price}: {inside} of {t.notna().sum()} liquidations inside 20:50–21:30 UTC")
minutes_10x

10x by mark: 1440 of 1440 liquidations inside 20:50–21:30 UTC
10x by last: 1440 of 1440 liquidations inside 20:50–21:30 UTC


,mark,last
10-10 21:13,863,1326
10-10 21:16,245,98
10-10 21:17,332,16
total,1440,1440
